In [1]:
import pandas as pd
import pdfplumber
import re

In [2]:
PDF_PATH = 'Workcover-insurance-industry-rates-and-industry-claims-cost-rates-2025-06.pdf'

pdf = pdfplumber.open(PDF_PATH)
print(f'Total pages: {len(pdf.pages)}')

Total pages: 20


In [3]:
# Extract all text lines from the PDF
all_lines = []
for page in pdf.pages:
    text = page.extract_text()
    if text:
        for line in text.split('\n'):
            all_lines.append(line.strip())

print(f'Total lines extracted: {len(all_lines)}')

Total lines extracted: 692


In [4]:
# Parse data rows.
# Most lines match: WIC_CODE Description RATE% RATE%
# Some descriptions wrap across two lines, e.g.:
#   "Corrugated Paperboard and Paperboard Container"
#   "C15210 Manufacturing 0.910% 4.053%"
# For these, the orphan description line precedes the WIC line and
# the WIC line's description is just the tail end (e.g. "Manufacturing").

full_pattern = re.compile(r'^([A-Z]\d{5})\s+(.+?)\s+(\d+\.\d+%)\s+(\d+\.\d+%)$')
wic_code_pattern = re.compile(r'^[A-Z]\d{5}\s+')

rows = []
pending_prefix = None

for line in all_lines:
    if not line:
        pending_prefix = None
        continue

    m = full_pattern.match(line)
    if m:
        wic_code = m.group(1)
        description = m.group(2)
        claims_cost_rate = m.group(3)
        industry_rate = m.group(4)

        # If there was a pending prefix from a wrapped description, prepend it
        if pending_prefix:
            joined = pending_prefix + ' ' + description
            # Fix hyphen-space artifacts from line wraps (e.g. "Non- Public" -> "Non-Public")
            description = re.sub(r'- ', '-', joined)
            pending_prefix = None

        rows.append({
            'wic_code': wic_code,
            'description': description,
            'claims_cost_rate_pct': float(claims_cost_rate.replace('%', '')),
            'industry_rate_pct': float(industry_rate.replace('%', '')),
        })
    elif not wic_code_pattern.match(line):
        # Not a WIC data line — could be a wrapped description prefix
        # Only treat it as a prefix if it looks like plain text (not a header/footer)
        if (not re.search(r'(Victoria Government|WorkCover Industry|Industry Claims|'
                          r'Classification|Cost Rate|S 275|This page|Government Printer|'
                          r'copyright|How To|Telephone|email|Price Code|IVE Group|'
                          r'No\. S 275|By Authority|Workplace Injury|WORKCOVER PREMIUMS|'
                          r'Specific Values|Notice is hereby|the Premiums Order|'
                          r'following values|premium period|SPECIAL|default industry|'
                          r'subclause|For the purposes|Address all|Melbourne|'
                          r'Richmond|Ground Floor|Level 2|State of|Mail Sales|'
                          r'reproduced by|provisions of|enquiries to)', line)
            and not re.match(r'^\d+$', line)
            and not re.match(r'^\d+ S \d+', line)
            and not re.match(r'^S \d+ ', line)
            and len(line) > 5):
            pending_prefix = line
        else:
            pending_prefix = None
    else:
        pending_prefix = None

df = pd.DataFrame(rows)
print(f'{len(df)} rows parsed')
df.head()

519 rows parsed


,wic_code,description,claims_cost_rate_pct,industry_rate_pct
0,A01110,Nursery Production (Under Cover),2.128,3.568
1,A01120,Nursery Production (Outdoors),1.031,3.534
2,A01130,Turf Growing,2.358,3.275
3,A01140,Floriculture Production (Under Cover),1.449,3.378
4,A01150,Floriculture Production (Outdoors),2.657,3.273


In [5]:
# Extract the ANZSIC division letter and numeric code from the WIC code
# VIC WIC codes are formatted as: Division letter + 5 digits (e.g. A01110)
df['division'] = df['wic_code'].str[0]
df['anzsic_class_code'] = df['wic_code'].str[1:5].astype(int).astype(str).str.zfill(4)

print(f"Divisions: {sorted(df['division'].unique())}")
print(f"\nRows per division:")
print(df.groupby('division')['wic_code'].count().sort_values(ascending=False))

Divisions: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S']

Rows per division:
division
C    143
A     49
F     39
G     37
J     24
E     24
I     24
S     22
Q     20
R     19
B     16
P     16
M     16
K     15
O     14
D     13
N     12
L     10
H      6
Name: wic_code, dtype: int64


In [6]:
df.describe()

,claims_cost_rate_pct,industry_rate_pct
count,519.000000,519.000000
mean,1.036863,2.822875
std,1.062442,1.919089
min,0.169000,0.360000
25%,0.287000,1.557000
50%,0.794000,2.607000
75%,1.388500,3.556500
max,10.266000,18.000000


In [7]:
# Spot-check some wrapped descriptions to make sure they parsed correctly
spot_checks = ['C15210', 'C13200', 'C14920', 'C17010', 'O77148', 'P80238', 'Q84018']
df[df['wic_code'].isin(spot_checks)][['wic_code', 'description']]

,wic_code,description
93,C13200,"Leather Tanning, Fur Dressing and Leather Prod..."
105,C14920,Wooden Structural Fitting and Component Manufa...
110,C15210,Corrugated Paperboard and Paperboard Container...
118,C17010,Petroleum Refining and Petroleum Fuel Manufact...
439,O77148,"Correctional and Detention Services, Non-Publi..."
448,P80238,"Combined Primary and Secondary Education, Non-..."
459,Q84018,"Hospitals (Except Psychiatric Hospitals), Non-..."


In [8]:
df.head()

,wic_code,description,claims_cost_rate_pct,industry_rate_pct,division,anzsic_class_code
0,A01110,Nursery Production (Under Cover),2.128,3.568,A,0111
1,A01120,Nursery Production (Outdoors),1.031,3.534,A,0112
2,A01130,Turf Growing,2.358,3.275,A,0113
3,A01140,Floriculture Production (Under Cover),1.449,3.378,A,0114
4,A01150,Floriculture Production (Outdoors),2.657,3.273,A,0115


In [9]:
# Export to CSV and Parquet
df.to_csv('VIC_WIC.csv', index=False)
df.to_parquet('VIC_WIC.parquet', index=False)
print('Saved VIC_WIC.csv')
print('Saved VIC_WIC.parquet')

Saved VIC_WIC.csv
Saved VIC_WIC.parquet
